# DXY Signal Evaluation
`apr_jan_mapped_w_macro.csv` — Apr 2025 + Jan 2026 articles

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("data/apr_jan_mapped_w_macro.csv")

relevant = df[df["is_relevant"] == True].copy()
critical = df[df["is_critical"] == True].copy()
dir_crit = critical[critical["direction"].isin(["up","down"])].copy()

horizons = ["pct_5m", "pct_15m", "pct_1h", "pct_4h", "pct_1d"]
sd = {h: df[h].std() for h in horizons}

print(f"Total articles : {len(df):,}")
print(f"Relevant       : {len(relevant):,}  ({len(relevant)/len(df)*100:.1f}%)")
print(f"Critical       : {len(critical):,}  ({len(critical)/len(relevant)*100:.1f}% of relevant)")
print(f"Critical w/ up/down direction: {len(dir_crit):,}")
print()
print("SDs  5m={:.4f}%  15m={:.4f}%  1h={:.4f}%  4h={:.4f}%  1d={:.4f}%".format(
    sd["pct_5m"], sd["pct_15m"], sd["pct_1h"], sd["pct_4h"], sd["pct_1d"]))

Total articles : 1,113
Relevant       : 631  (56.7%)
Critical       : 261  (41.4% of relevant)
Critical w/ up/down direction: 248

SDs  5m=0.0410%  15m=0.0644%  1h=0.1122%  4h=0.2285%  1d=0.5929%


## 1. Directional Accuracy by Criticality Level
Filtered to rows where direction ∈ {up, down} and intraday price was matched.

In [2]:
def dir_accuracy(subset, horizon):
    s = subset[subset["direction"].isin(["up","down"]) & subset[horizon].notna()].copy()
    actual = s[horizon].apply(lambda x: "up" if x > 0 else "down")
    hits = (actual == s["direction"]).sum()
    return hits, len(s)

levels = ["high", "medium", "low"]
rows = []
for lvl in levels + ["all_critical"]:
    sub = dir_crit if lvl == "all_critical" else critical[
        (critical["criticality_level"] == lvl) & critical["direction"].isin(["up","down"])]
    row = {"criticality": lvl}
    for h in horizons:
        hits, n = dir_accuracy(sub, h)
        row[h] = f"{hits/n*100:.1f}% ({hits}/{n})" if n else "n/a"
    rows.append(row)

display(pd.DataFrame(rows).set_index("criticality"))

,pct_5m,pct_15m,pct_1h,pct_4h,pct_1d
criticality,,,,,
high,54.5% (55/101),54.5% (55/101),38.8% (38/98),44.8% (43/96),39.5% (32/81)
medium,46.4% (45/97),54.2% (52/96),45.7% (43/94),53.9% (48/89),47.6% (40/84)
low,n/a,n/a,n/a,n/a,n/a
all_critical,50.5% (100/198),54.3% (107/197),42.2% (81/192),49.2% (91/185),43.6% (72/165)


## 2. Directional Accuracy by Direction Confidence

In [3]:
rows = []
for conf in ["high", "medium", "low"]:
    sub = dir_crit[dir_crit["direction_confidence"] == conf]
    row = {"dir_confidence": conf, "n": len(sub)}
    for h in horizons:
        hits, n = dir_accuracy(sub, h)
        row[h] = f"{hits/n*100:.1f}% ({hits}/{n})" if n else "n/a"
    rows.append(row)

display(pd.DataFrame(rows).set_index("dir_confidence"))

,n,pct_5m,pct_15m,pct_1h,pct_4h,pct_1d
dir_confidence,,,,,,
high,91,56.6% (43/76),56.6% (43/76),39.7% (29/73),45.3% (34/75),38.7% (24/62)
medium,131,47.1% (48/102),53.5% (54/101),40.4% (40/99),50.0% (46/92),47.1% (41/87)
low,26,45.0% (9/20),50.0% (10/20),60.0% (12/20),61.1% (11/18),43.8% (7/16)


## 3. Table Usage
Critical articles that used the historical response table vs. didn't.

In [4]:
tbl = critical.groupby("table_used").agg(n=("id","count"),
    pct_5m_mean=("pct_5m","mean"), pct_15m_mean=("pct_15m","mean"),
    pct_1h_mean=("pct_1h","mean"), pct_4h_mean=("pct_4h","mean"),
).round(4)
tbl.index = tbl.index.map({True: "table_used", False: "no_table"})
display(tbl)

print()
for label, mask in [("table_used", critical["table_used"]==True), ("no_table", critical["table_used"]==False)]:
    sub = critical[mask & critical["direction"].isin(["up","down"])]
    for h in ["pct_5m", "pct_15m", "pct_1h"]:
        hits, n = dir_accuracy(sub, h)
        acc = f"{hits/n*100:.1f}% ({hits}/{n})" if n else "n/a"
        print(f"  {label}  {h}: {acc}")
    print()

,n,pct_5m_mean,pct_15m_mean,pct_1h_mean,pct_4h_mean
table_used,,,,,
no_table,233,0.0013,0.0025,0.0196,0.0129
table_used,28,0.0204,0.0168,-0.0204,0.0036



  table_used  pct_5m: 50.0% (13/26)
  table_used  pct_15m: 53.8% (14/26)
  table_used  pct_1h: 46.2% (12/26)

  no_table  pct_5m: 50.6% (87/172)
  no_table  pct_15m: 54.4% (93/171)
  no_table  pct_1h: 41.6% (69/166)



## 4. Mean |DXY %| Change by Criticality Level

In [5]:
rows = []
for lvl in levels + ["all_relevant", "all"]:
    if lvl == "all_relevant":
        sub = relevant[relevant["pct_1h"].notna()]
    elif lvl == "all":
        sub = df[df["pct_1h"].notna()]
    else:
        sub = df[(df["criticality_level"] == lvl) & df["pct_1h"].notna()]
    row = {"group": lvl, "n": len(sub)}
    for h in horizons:
        row[f"|{h}|"] = round(sub[h].abs().mean(), 4)
    rows.append(row)

display(pd.DataFrame(rows).set_index("group"))

,n,|pct_5m|,|pct_15m|,|pct_1h|,|pct_4h|,|pct_1d|
group,,,,,,
high,98,0.0396,0.0670,0.1080,0.2320,0.5772
medium,105,0.0266,0.0435,0.0774,0.1659,0.4739
low,310,0.0238,0.0383,0.0758,0.1563,0.4162
all_relevant,513,0.0274,0.0449,0.0823,0.1732,0.4595
all,828,0.0258,0.0433,0.0773,0.1625,0.4090


## 5. Moves ≥ 1SD / 2SD Captured in High & Medium Critical

In [6]:
def sd_capture(subset, horizon, threshold):
    s = subset[subset["direction"].isin(["up","down"]) & subset[horizon].notna()].copy()
    s["actual_dir"] = s[horizon].apply(lambda x: "up" if x > 0 else "down")
    above = s[s[horizon].abs() >= threshold]
    hits  = (above["actual_dir"] == above["direction"]).sum()
    return hits, len(above)

rows = []
for lvl in ["high", "medium"]:
    sub = critical[critical["criticality_level"] == lvl]
    for h in horizons:
        h1, n1 = sd_capture(sub, h, sd[h])
        h2, n2 = sd_capture(sub, h, sd[h]*2)
        rows.append({
            "criticality": lvl,
            "horizon":     h,
            "1SD thresh":  f"{sd[h]:.4f}%",
            "≥1SD":        f"{h1}/{n1} = {h1/n1*100:.1f}%" if n1 else "n/a",
            "2SD thresh":  f"{sd[h]*2:.4f}%",
            "≥2SD":        f"{h2}/{n2} = {h2/n2*100:.1f}%" if n2 else "n/a",
        })

display(pd.DataFrame(rows).set_index(["criticality","horizon"]))

1SD thresh           ≥1SD 2SD thresh           ≥2SD
criticality horizon                                                    
high        pct_5m     0.0410%  17/30 = 56.7%    0.0820%  10/13 = 76.9%
            pct_15m    0.0644%  21/43 = 48.8%    0.1289%   4/10 = 40.0%
            pct_1h     0.1122%  17/36 = 47.2%    0.2245%   6/10 = 60.0%
            pct_4h     0.2285%  23/40 = 57.5%    0.4569%   7/12 = 58.3%
            pct_1d     0.5929%   9/28 = 32.1%    1.1859%   3/12 = 25.0%
medium      pct_5m     0.0410%   9/20 = 45.0%    0.0820%    4/6 = 66.7%
            pct_15m    0.0644%   9/21 = 42.9%    0.1289%    3/5 = 60.0%
            pct_1h     0.1122%  17/25 = 68.0%    0.2245%    3/6 = 50.0%
            pct_4h     0.2285%  13/22 = 59.1%    0.4569%    6/8 = 75.0%
            pct_1d     0.5929%  12/22 = 54.5%    1.1859%   5/10 = 50.0%

## 6. Mann-Whitney U Test: Critical vs Non-Critical Absolute Moves
One-sided test (H₁: critical articles produce larger absolute DXY moves).

In [7]:
from scipy.stats import mannwhitneyu

rows = []
for h in horizons:
    crit_vals    = critical[h].dropna().abs()
    noncrit_vals = df[df["is_critical"] != True][h].dropna().abs()
    stat, p = mannwhitneyu(crit_vals, noncrit_vals, alternative="greater")
    rows.append({
        "horizon":        h,
        "n_critical":     len(crit_vals),
        "n_non_critical": len(noncrit_vals),
        "mean |crit|":    round(crit_vals.mean(), 4),
        "mean |non-crit|":round(noncrit_vals.mean(), 4),
        "U stat":         round(stat, 1),
        "p-value":        round(p, 4),
        "sig (p<0.05)":   "yes" if p < 0.05 else "no",
    })

mw_df = pd.DataFrame(rows).set_index("horizon")
display(mw_df)

,n_critical,n_non_critical,mean |crit|,mean |non-crit|,U stat,p-value,sig (p<0.05)
horizon,,,,,,,
pct_5m,209,638,0.0325,0.0233,77310.5,0.0003,yes
pct_15m,208,635,0.0540,0.0392,76909.0,0.0002,yes
pct_1h,203,625,0.0922,0.0725,70656.0,0.0074,yes
pct_4h,196,569,0.2016,0.1504,66468.0,0.0000,yes
pct_1d,172,492,0.5468,0.3705,48948.0,0.0011,yes
